# ZeroTuning: Reproduction Notebook

Reproduces the core mechanism from:
> **ZeroTuning: Unlocking the Initial Token's Power to Enhance Large Language Models Without Training**  
> https://arxiv.org/abs/2505.11739  
> GitHub: https://github.com/FeijiangHan/ZeroTuning

**What this notebook does:**
1. Installs dependencies (exact versions required)
2. Downloads the patched LLaMA attention file and evaluation framework
3. Loads Meta-Llama-3.1-8B-Instruct with 8-bit quantization
4. Demonstrates the core mechanism with a single example (baseline vs ZeroTuning)
5. Runs full SST2 evaluation (500 samples) at baseline and ZeroTuning
6. Sweeps the `rate` hyperparameter to reproduce the sensitivity curve

**Runtime requirement:** GPU (T4 or better, ~10 GB VRAM with 8-bit quant)

## Cell 1 — Install Dependencies

`transformers==4.53.2` is **mandatory** — the patched modeling file monkey-patches internal LlamaAttention and is version-locked to this release.

In [13]:
!pip install -q \
    transformers==4.53.2 \
    datasets==4.1.1 \
    bitsandbytes==0.47.0 \
    accelerate \
    gdown==5.2.0

print("Done. Restart runtime if prompted by Colab.")

Done. Restart runtime if prompted by Colab.


## Cell 2 — Download ZeroTuning Files

Downloads 4 artifacts from the official repository:
- `modelling_llama_open.py` — patched LlamaAttention with ZeroTuning_Attention()
- `eval_classification_final.py` — 23-dataset evaluation framework
- `eval_config.json` — evaluation configuration
- `data.tar.gz` — SST2 and other datasets (extracts to `./data/`)

In [14]:
import gdown
import os
import tarfile
import zipfile

os.makedirs("data", exist_ok=True)
os.makedirs("outputs", exist_ok=True)

# Official GDrive IDs from the ZeroTuning README
files = {
    "modelling_llama_open.py": "1JYr9Do94hfzc91NyxKBSJCwsTNw5ygK6",
    "eval_classification_final.py": "17PhF8wGp9X5puN_kr0WzW7r5_ynlShXs",
    "eval_config.json": "1ZNbNV_ePNckVuNbkzQjJAqJoODy-GSW8",
    "data.tar.gz": "1LjcvWQ84JqZQKMQrsxxIVnv0uug8hKWP",
}

for filename, gdrive_id in files.items():
    if not os.path.exists(filename):
        print(f"Downloading {filename}...")
        gdown.download(id=gdrive_id, output=filename, quiet=False)
    else:
        print(f"{filename} already exists, skipping.")

# Extract dataset archive — GDrive sometimes serves a ZIP despite the .tar.gz name
DATA_ARCHIVE = "data.tar.gz"
if os.path.exists(DATA_ARCHIVE) and not os.path.exists("data/sst2"):
    print("Extracting datasets...")
    # Detect actual format by magic bytes
    with open(DATA_ARCHIVE, "rb") as f:
        magic = f.read(4)

    if magic[:2] == b"PK":  # ZIP magic bytes
        print("  (detected ZIP format)")
        with zipfile.ZipFile(DATA_ARCHIVE, "r") as z:
            z.extractall(".")
    elif magic[:2] == b"\x1f\x8b":  # gzip magic bytes
        print("  (detected gzip/tar.gz format)")
        with tarfile.open(DATA_ARCHIVE, "r:gz") as tar:
            tar.extractall(".")
    else:
        print("  (unknown format, trying tarfile auto-detect)")
        with tarfile.open(DATA_ARCHIVE, "r:*") as tar:
            tar.extractall(".")

    print("Extraction complete.")

print("\nFiles ready:")
for f in ["modelling_llama_open.py", "eval_classification_final.py", "eval_config.json"]:
    status = "OK" if os.path.exists(f) else "MISSING"
    print(f"  {f}: {status}")

modelling_llama_open.py already exists, skipping.
eval_classification_final.py already exists, skipping.
eval_config.json already exists, skipping.
data.tar.gz already exists, skipping.
Extracting datasets...
  (detected ZIP format)
Extraction complete.

Files ready:
  modelling_llama_open.py: OK
  eval_classification_final.py: OK
  eval_config.json: OK


## Cell 3 — Set HuggingFace Token

Meta-Llama-3.1-8B-Instruct requires a HuggingFace token with access granted at:
https://huggingface.co/meta-llama/Meta-Llama-3.1-8B-Instruct

In [15]:
import os
from google.colab import userdata  # type: ignore

# Option A: use Colab Secrets (Colab > left sidebar > key icon > add HF_TOKEN)
try:
    HF_TOKEN = userdata.get("HF_TOKEN")
    print("HF_TOKEN loaded from Colab Secrets.")
except Exception:
    # Option B: paste directly (less secure)
    HF_TOKEN = "hf_YOUR_TOKEN_HERE"
    print("Using hardcoded token — consider using Colab Secrets instead.")

os.environ["HF_TOKEN"] = HF_TOKEN

Using hardcoded token — consider using Colab Secrets instead.


## Cell 4 — Load Model with Patched Attention

Importing `modelling_llama_open` monkey-patches `transformers.models.llama.modeling_llama.LlamaAttention.forward`
to intercept the `input_len` kwarg and call `ZeroTuning_Attention()` after softmax.

In [ ]:
import torch
import importlib.util, sys
import gc

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU를 찾을 수 없습니다.\n"
        "Colab 상단 메뉴 → 런타임 → 런타임 유형 변경 → T4 GPU 선택 후 재시작하세요."
    )
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM 전체: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

for _var in ["model", "tokenizer"]:
    if _var in globals():
        del globals()[_var]
gc.collect()
torch.cuda.empty_cache()
print(f"VRAM 사용 (정리 후): {torch.cuda.memory_allocated() / 1e9:.1f} GB")

# modelling_llama_open.py 로드
spec = importlib.util.spec_from_file_location("modelling_llama_open", "modelling_llama_open.py")
mod = importlib.util.module_from_spec(spec)
sys.modules["modelling_llama_open"] = mod
spec.loader.exec_module(mod)
print("modelling_llama_open loaded.")

# 원래 레포 방식: AutoModelForCausalLM 대신 파일의 LlamaForCausalLM 직접 사용
# eval_classification_final.py의 load_model()과 동일한 방식
LlamaForCausalLM = mod.LlamaForCausalLM

from transformers import AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "meta-llama/Meta-Llama-3.1-8B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_enable_fp32_cpu_offload=True,
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    padding_side="left",
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading model via modelling_llama_open.LlamaForCausalLM ...")
model = LlamaForCausalLM.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    torch_dtype=torch.float16,
    device_map="auto",
    quantization_config=bnb_config,
)
model.eval()

print(f"\nAttention class: {type(model.model.layers[0].self_attn).__name__}")
print(f"Model loaded on: {next(model.parameters()).device}")
print(f"GPU memory used: {torch.cuda.memory_allocated() / 1e9:.1f} GB")


In [ ]:
import torch
from transformers.generation.utils import GenerationMixin

# modelling_llama_open.LlamaForCausalLM은 prepare_inputs_for_generation에서
# input_len을 이미 처리한다. _validate_model_kwargs가 혹시 거부할 경우를 대비해
# 안전망으로만 유지.

_orig_validate = GenerationMixin._validate_model_kwargs
def _validate(self, model_kwargs):
    _orig_validate(self, {k: v for k, v in model_kwargs.items() if k != "input_len"})
GenerationMixin._validate_model_kwargs = _validate

print("Safety patch applied.")


def run_inference(prompt, rate, target_layers=None, target_heads=None):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=1,
            do_sample=False,
            num_beams=1,
            pad_token_id=tokenizer.pad_token_id,
            input_len=(0, 0, rate, target_layers, target_heads),
        )
    return tokenizer.decode(out[0][-1]).strip()


review = "a stirring , funny and finally transporting re-imagining of beauty and the beast ."
prompt = (
    f"Review: {review}\n"
    "Sentiment (answer with only one word - positive or negative):"
)

baseline = run_inference(prompt, rate=1.0)
enhanced = run_inference(prompt, rate=4.0)

print(f"Review    : {review}")
print(f"True label: positive")
print(f"Baseline   (rate=1.0): {baseline}")
print(f"ZeroTuning (rate=4.0): {enhanced}")
print()
if baseline.lower() != enhanced.lower():
    print("ZeroTuning is changing the output — patch working.")
else:
    print("Both outputs same — check if patch is applied correctly.")


In [ ]:
import importlib.util, sys, os, random as _random
import datasets as _hf_datasets
from datasets import load_from_disk as _load_from_disk

spec = importlib.util.spec_from_file_location("eval_cls", "eval_classification_final.py")
eval_mod = importlib.util.module_from_spec(spec)
sys.modules["eval_cls"] = eval_mod
spec.loader.exec_module(eval_mod)

# ── patch load_mc_dataset ────────────────────────────────────────────────────
_HF_FALLBACK = {
    "sst2": ("glue",            "sst2", "sentence"),
    "sst5": ("SetFit/sst5",     None,   "text"),
    "MR":   ("rotten_tomatoes", None,   "text"),
}

_orig_load_mc = eval_mod.load_mc_dataset

def _patched_load_mc(data_path, dataset_name, split="validation", num_samples=-1):
    local_path = os.path.join(data_path, dataset_name)

    # 1) load_from_disk 우선 (이전에 save_to_disk로 저장한 경우)
    try:
        ds_dict = _load_from_disk(local_path)
        split_name = eval_mod.DATASET_CONFIGS[dataset_name]["split_names"].get(split, split)
        dataset = ds_dict[split_name]
        if num_samples > 0 and num_samples < len(dataset):
            dataset = dataset.select(_random.sample(range(len(dataset)), num_samples))
        print(f"Loaded {dataset_name} from disk cache ({local_path}).")
        return dataset
    except Exception:
        pass

    # 2) 기존 로직 시도 (다른 데이터셋용 Arrow/JSON 파일)
    try:
        result = _orig_load_mc(data_path, dataset_name, split=split, num_samples=num_samples)
        if result is not None:
            return result
    except Exception:
        pass

    # 3) HuggingFace에서 다운로드
    if dataset_name not in _HF_FALLBACK:
        raise FileNotFoundError(f"Cannot load {dataset_name}: no local cache and no HF fallback.")
    hub_path, hub_cfg, src_col = _HF_FALLBACK[dataset_name]
    print(f"Downloading {dataset_name} from HuggingFace ({hub_path})...")
    ds = _hf_datasets.load_dataset(hub_path, hub_cfg) if hub_cfg else _hf_datasets.load_dataset(hub_path)

    for s in ds:
        if src_col != "text" and src_col in ds[s].column_names:
            ds[s] = ds[s].rename_column(src_col, "text")
        extra = [c for c in ds[s].column_names if c not in {"text", "label"}]
        if extra:
            ds[s] = ds[s].remove_columns(extra)

    os.makedirs(local_path, exist_ok=True)
    ds.save_to_disk(local_path)
    print(f"Saved {dataset_name} to {local_path}.")

    split_name = eval_mod.DATASET_CONFIGS[dataset_name]["split_names"].get(split, split)
    dataset = ds[split_name]
    if num_samples > 0 and num_samples < len(dataset):
        dataset = dataset.select(_random.sample(range(len(dataset)), num_samples))
    return dataset

eval_mod.load_mc_dataset = _patched_load_mc
# ─────────────────────────────────────────────────────────────────────────────

EVAL_KWARGS_BASE = dict(
    model_path=MODEL_ID,
    model_type="llama",
    dataset_name="sst2",
    data_path="./data",
    num_samples=500,
    few_shot_number=0,
    output_dir="outputs",
    verbose=False,
    use_8bit=True,
    heads=None,
    layers=None,
    exploring_mode=None,
    rate_min=None,
    rate_max=None,
    rate_step=None,
)

results = {}

for rate in [1.0, 4.0]:
    label = "Baseline" if rate == 1.0 else "ZeroTuning"
    print(f"\n{'='*50}")
    print(f"Running {label} (rate={rate}) on SST2 (500 samples)...")
    print(f"{'='*50}")

    acc = eval_mod.ZeroTuning(
        model=model,
        tokenizer=tokenizer,
        rate=rate,
        **EVAL_KWARGS_BASE,
    )
    results[label] = acc
    print(f"{label} accuracy: {acc:.2%}")

print("\n" + "="*50)
print("RESULTS SUMMARY")
print("="*50)
for label, acc in results.items():
    print(f"  {label}: {acc:.2%}")

if "Baseline" in results and "ZeroTuning" in results:
    rel_gain = (results["ZeroTuning"] - results["Baseline"]) / results["Baseline"] * 100
    abs_gain = (results["ZeroTuning"] - results["Baseline"]) * 100
    print(f"  Absolute gain: +{abs_gain:.1f}pp")
    print(f"  Relative gain: +{rel_gain:.1f}%")


## Cell 7 — Rate Sweep (SST2)

Sweeps `rate` from 1.0 → 6.0 to reproduce the rate sensitivity figure from the paper.
This shows how accuracy peaks at an optimal rate and degrades beyond it.

In [ ]:
import matplotlib.pyplot as plt

rates = [1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0, 5.5, 6.0]
accuracies = []

for rate in rates:
    print(f"rate={rate:.1f} ...", end=" ", flush=True)
    acc = eval_mod.ZeroTuning(
        model=model,
        tokenizer=tokenizer,
        rate=rate,
        **{**EVAL_KWARGS_BASE, "num_samples": -1},
    )
    accuracies.append(acc)
    print(f"{acc:.2%}")

print("\nRate sweep complete.")

plt.figure(figsize=(8, 4))
plt.plot(rates, [a * 100 for a in accuracies], marker="o", linewidth=2, color="steelblue")
plt.axhline(accuracies[0] * 100, linestyle="--", color="gray", label="Baseline (rate=1.0)")
best_idx = accuracies.index(max(accuracies))
plt.scatter([rates[best_idx]], [max(accuracies) * 100], color="red", zorder=5,
            label=f"Best: rate={rates[best_idx]}, acc={max(accuracies):.2%}")
plt.xlabel("Rate", fontsize=12)
plt.ylabel("Accuracy (%)", fontsize=12)
plt.title("ZeroTuning Rate Sensitivity — SST2 (Llama-3.1-8B-Instruct)", fontsize=13)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("outputs/rate_sweep_sst2.png", dpi=150)
plt.show()
print("Figure saved to outputs/rate_sweep_sst2.png")


## Cell 8 — (Optional) Layer & Head Targeting

Instead of modifying all layers and heads, ZeroTuning can target specific ones.
This cell demonstrates the API for targeted application.

In [ ]:
# Example: apply ZeroTuning only to the last 8 layers, all heads
num_layers = model.config.num_hidden_layers  # 32 for Llama-3.1-8B
target_layers = list(range(num_layers - 8, num_layers))

print(f"Model has {num_layers} layers.")
print(f"Targeting layers: {target_layers}")

result_targeted = run_inference(prompt, rate=4.0, target_layers=target_layers, target_heads=None)
result_all = run_inference(prompt, rate=4.0, target_layers=None, target_heads=None)

print(f"\nMotivating example:")
print(f"  ZeroTuning (all layers) : {result_all}")
print(f"  ZeroTuning (last 8 only): {result_targeted}")

## Cell 9 — (Optional) Run eval_classification_final.py via CLI

Alternative: run the evaluation script directly from the command line for any supported dataset.

In [ ]:
# Example CLI usage — edit arguments as needed
!python eval_classification_final.py \
    --model_path meta-llama/Meta-Llama-3.1-8B-Instruct \
    --dataset_name sst2 \
    --data_path ./data \
    --num_samples 500 \
    --rate 4.0 \
    --use_8bit \
    --output_dir outputs